In [1]:
# =========================================
# BEAD THREADING PRECISION (FULL PIPELINE)
# =========================================
import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math

In [2]:
# MediaPipe Setup (Hands + Pose)

mpHands = mp.solutions.hands
hands = mpHands.Hands(min_detection_confidence=0.6)

mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils

DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# Utility Functions

def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [4]:
# -------------------------------
# QUALITY FUNCTION (FULL BIO)
# -------------------------------
def compute_hand_quality(hand_data):

    thumb = np.array(hand_data['thumb'])
    index = np.array(hand_data['index'])
    middle = np.array(hand_data['middle'])
    ring = np.array(hand_data['ring'])
    pinky = np.array(hand_data['pinky'])

    n = min(len(thumb), len(index), len(middle), len(ring), len(pinky))
    if n < 5:
        return 1

    thumb, index, middle, ring, pinky = thumb[:n], index[:n], middle[:n], ring[:n], pinky[:n]

    # -------- PINCER --------
    pinch = np.linalg.norm(thumb - index, axis=1)

    # -------- GRIP TYPE --------
    grip = np.mean([
        np.linalg.norm(thumb - middle, axis=1),
        np.linalg.norm(thumb - ring, axis=1),
        np.linalg.norm(thumb - pinky, axis=1)
    ])

    # -------- FINGER SPREAD --------
    spread = np.std([index[:,1], middle[:,1], ring[:,1], pinky[:,1]])

    # -------- SMOOTHNESS --------
    vel = np.diff(index[:,1])
    accel = np.diff(vel) if len(vel) > 1 else np.array([0])
    jerk = np.diff(accel) if len(accel) > 1 else np.array([0])
    jerk_val = safe_std(jerk)

    # -------- STABILITY --------
    stability = safe_std(index[:,0]) + safe_std(index[:,1])

    # -------- EFFICIENCY --------
    path = np.sum(np.linalg.norm(np.diff(index, axis=0), axis=1))
    disp = np.linalg.norm(index[-1] - index[0]) + 1e-6
    efficiency = disp / path

    score = 0

    if safe_mean(pinch) < 0.03: score += 2
    elif safe_mean(pinch) < 0.06: score += 1

    if grip < 0.1: score += 1
    if spread > 0.01: score += 1

    if jerk_val < 0.005: score += 2
    elif jerk_val < 0.01: score += 1

    if stability < 0.01: score += 1
    if efficiency > 0.7: score += 1

    return max(1, min(5, round(score)))

In [5]:
def sync_cal(dom_y, sup_y):
    min_len = min(len(dom_y), len(sup_y))
    if min_len > 5:
        return safe_mean(np.abs(np.array(dom_y[:min_len]) - np.array(sup_y[:min_len])))
    else:
        return 0.1

In [6]:
#adjusted quality score
def quality_cal(sync, q_dom, q_sup):
    if sync < 0.02:
        return round((q_dom + q_sup) / 2)
    elif sync < 0.05:
        return round((q_dom + q_sup) / 2) - 1
    else:
        return max(1, round((q_dom + q_sup) / 2) - 2)


In [7]:
def final_block_score(blocks, quality, age_group):

    # -------------------------------
    # AGE NORMS (BLOCKS)
    # -------------------------------
    norms = {
        "2.5-3": (3, 4),
        "3-4": (5, 6),
        "4-5": (7, 8)
    }

    low, high = norms.get(age_group, (5,6))

    # -------------------------------
    # FLOOR CONDITION
    # -------------------------------
    if blocks <= 1 or quality < 2:
        return 1

    # -------------------------------
    # SCORE 5 (EXCELLENT)
    # >= above norm + high quality
    # -------------------------------
    if blocks >= high + 1 and quality >= 4.5:
        return 5

    # -------------------------------
    # SCORE 4 (ABOVE AVG)
    # slightly below + good quality
    # -------------------------------
    if blocks >= high and 4.0 <= quality < 4.5:
        return 4

    # -------------------------------
    # SCORE 3 (AT NORM)
    # -------------------------------
    if blocks >= low and 3.0 <= quality < 4.0:
        return 3

    # -------------------------------
    # SCORE 2 (BELOW EXPECTATION)
    # -------------------------------
    if blocks >= low - 2 and 2.0 <= quality < 3.0:
        return 2

    # -------------------------------
    # DEFAULT
    # -------------------------------
    return 1

In [8]:
def final_bead_score(beads, quality, age_group):

    # -------------------------------
    # AGE NORMS (BEADS)
    # -------------------------------
    norms = {
        "2.5-3": (4, 6),
        "3-4": (7, 9),
        "4-5": (10, 12)
    }

    low, high = norms.get(age_group, (7,9))

    # -------------------------------
    # FLOOR CONDITION
    # -------------------------------
    if beads <= 3 or quality < 2:
        return 1

    # -------------------------------
    # SCORE 5 (EXCELLENT)
    # -------------------------------
    if beads >= high and quality >= 4.5:
        return 5

    # -------------------------------
    # SCORE 4
    # -------------------------------
    if beads >= high - 1 and 4.0 <= quality < 4.5:
        return 4

    # -------------------------------
    # SCORE 3 (AT NORM)
    # -------------------------------
    if beads >= low and 3.0 <= quality < 4.0:
        return 3

    # -------------------------------
    # SCORE 2 (BELOW)
    # -------------------------------
    if beads >= low - 3 and 2.0 <= quality < 3.0:
        return 2

    # -------------------------------
    # DEFAULT
    # -------------------------------
    return 1

In [9]:
# -------------------------------
# MAIN FUNCTION
# -------------------------------
def bead_threading_precision(path, age_group="3-4"):

    cap = cv2.VideoCapture(path)

    # -------- HAND DATA --------
    dom_data = {k: [] for k in ['thumb','index','middle','ring','pinky','wrist']}
    sup_data = {k: [] for k in ['thumb','index','middle','ring','pinky','wrist']}

    velocities, accelerations, jerks = [], [], []
    dom_y, sup_y = [], []

    beads = 0
    drops = 0

    prev_y, prev_vel, prev_acc = None, None, None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                mpDraw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS)

                lm = hand_landmarks.landmark

                x = lm[8].x
                y = lm[8].y

                # -------- DOM / SUPPORT SPLIT --------
                target = dom_data if x > 0.5 else sup_data

                target['thumb'].append((lm[4].x, lm[4].y))
                target['index'].append((lm[8].x, lm[8].y))
                target['middle'].append((lm[12].x, lm[12].y))
                target['ring'].append((lm[16].x, lm[16].y))
                target['pinky'].append((lm[20].x, lm[20].y))
                target['wrist'].append((lm[0].x, lm[0].y))

                # -------- MOTION --------
                if prev_y is not None:
                    vel = y - prev_y
                    velocities.append(vel)

                    if prev_vel is not None:
                        acc = vel - prev_vel
                        accelerations.append(acc)

                        if prev_acc is not None:
                            jerk = acc - prev_acc
                            jerks.append(jerk)

                        prev_acc = acc
                    prev_vel = vel

                    # -------- EVENT DETECTION --------
                    if vel > 0.02:
                        beads += 1

                    if vel > 0.05:
                        drops += 1

                prev_y = y

                # -------- BILATERAL TRACK --------
                if x > 0.5:
                    dom_y.append(y)
                else:
                    sup_y.append(y)

        cv2.imshow("Bead Threading Precision", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

    # =========================
    # QUALITY SCORE
    q_dom = compute_hand_quality(dom_data)
    q_sup = compute_hand_quality(sup_data)

    # bilateral coordination penalty
    sync = sync_cal(dom_y, sup_y)

    #adjusted quality score
    quality = quality_cal(sync, q_dom, q_sup)
    quality = max(1, min(5, quality))

    # =========================
    # final score
    final = final_bead_score(beads, quality, age_group)

    # =========================
    # OUTPUT
    print("------ BEAD THREADING RESULT ------")
    print(f"Beads Threaded: {beads}")
    print(f"Drops: {drops}")
    print(f"Dominant Quality: {q_dom}")
    print(f"Support Quality: {q_sup}")
    print(f"Bilateral Sync: {sync:.3f}")
    print(f"Final Quality Score: {quality}")
    print(f"Final Score: {final}")

    return final, quality

In [10]:
path = "data/beads_FM_1.mp4"
print(bead_threading_precision(path))

C:\Users\nayan\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


------ BEAD THREADING RESULT ------
Beads Threaded: 1066
Drops: 708
Dominant Quality: 3
Support Quality: 3
Bilateral Sync: 0.168
Final Quality Score: 1
Final Score: 1
(1, 1)
